In [1]:
import datetime
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import json

from sklearn import datasets, ensemble, model_selection
from scipy.stats import anderson_ksamp

In [2]:
content = requests.get("https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip").content
with zipfile.ZipFile(io.BytesIO(content)) as arc:
    raw_data = pd.read_csv(arc.open("hour.csv"), header=0, sep=',', parse_dates=['dteday'])

In [3]:
raw_data.index = raw_data.apply(lambda row: datetime.datetime.combine(row.dteday.date(), datetime.time(row.hr)), axis=1)

In [4]:
raw_data.head()


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
2011-01-01 00:00:00,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
2011-01-01 01:00:00,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2011-01-01 02:00:00,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
2011-01-01 03:00:00,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
2011-01-01 04:00:00,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


### The Kolmogorov-Smirnov Test

“What is the probability that these two sets of samples were drawn from the same probability distribution?”


The null hypothesis is that the two samples come from the same distribution. The KS-test is applied to reject or accept it.

p-value: the probability of observing the data (or something more extreme) **given that the null hypothesis is true**. A low p-value indicates that the observed data is unlikely under the null hypothesis, leading to its rejection.

In [5]:
from scipy import stats
import random

#Significance level

p_value = 0.05
rejected = 0

numerical_features = ['temp', 'atemp', 'hum', 'windspeed', 'mnth', 'hr', 'weekday']
categorical_features = ['season', 'holiday', 'workingday']

reference = raw_data.loc['2011-01-01 00:00:00':'2011-01-28 23:00:00']
current = raw_data.loc['2011-01-29 00:00:00':'2011-02-28 23:00:00']

for col in numerical_features:
    test = stats.ks_2samp(reference[col], current[col])

    print(col, test)
    if test[1] < p_value:
        rejected += 1
        print("Column rejected", col)

print("We rejected ",rejected," columns in total out of {} columns".format(len(numerical_features)))

temp KstestResult(statistic=np.float64(0.3268630919426928), pvalue=np.float64(6.026276035162504e-32), statistic_location=np.float64(0.22), statistic_sign=np.int8(1))
Column rejected temp
atemp KstestResult(statistic=np.float64(0.3233995435947986), pvalue=np.float64(2.855460838157155e-31), statistic_location=np.float64(0.2273), statistic_sign=np.int8(1))
Column rejected atemp
hum KstestResult(statistic=np.float64(0.10336407541938417), pvalue=np.float64(0.0014918623865842317), statistic_location=np.float64(0.35), statistic_sign=np.int8(-1))
Column rejected hum
windspeed KstestResult(statistic=np.float64(0.07590999725436713), pvalue=np.float64(0.04055120974818981), statistic_location=np.float64(0.3284), statistic_sign=np.int8(1))
Column rejected windspeed
mnth KstestResult(statistic=np.float64(0.9026425591098748), pvalue=np.float64(1.472945287377825e-288), statistic_location=np.int64(1), statistic_sign=np.int8(1))
Column rejected mnth
hr KstestResult(statistic=np.float64(0.011077053260776